In [2]:
import torch                   # PyTorch：深度学习核心库，提供张量运算和自动求导
import torch.nn.functional as F # F 模块：包含各种不带可学习参数的函数（如交叉熵、softmax等）
import matplotlib.pyplot as plt # matplotlib：绘图库，用于可视化训练过程和结果
# 让图表直接嵌入 notebook 显示，无需调用 plt.show()
%matplotlib inline   

# Lecture 6: Building makemore — WaveNet 风格的层级融合网络

> 跟随 Karpathy 的 *Neural Networks: Zero to Hero* 系列第 6 讲。
> 核心升级：从 Lecture 4 的"全连接 MLP"升级为 **WaveNet 风格的层级架构**——相邻字符先两两融合，逐层向上汇聚，用更深、更有结构的方式处理长上下文。

**学习路线：**
1. **数据准备** — 扩大上下文窗口 `block_size=8`，用更长的历史预测下一个字符
2. **WaveNet 架构** — 用 `FlattenConsecutive` 实现二叉树式的层级融合，用类封装所有层（Embedding、Linear、BatchNorm1d、Tanh、FlattenConsecutive、Sequential）
3. **训练** — 200k 步 SGD + 阶梯式学习率衰减
4. **评估与采样** — 对比 train/dev loss，从模型中采样生成名字

## 1. 数据准备

与前几讲相同：读取 32k 个英文名字，构建字符↔索引映射（27 = 26 字母 + 1 个特殊标记 '.'），用滑动窗口切出训练样本。

**本讲的关键改动：** `block_size` 从 3 扩大到 **8**——模型现在能"看到"前 8 个字符来预测下一个。上下文越长，模型能捕捉的模式越丰富（比如能学到 "tion" 这种 4 字母后缀）。

In [3]:
# 读取姓名数据集，每行一个名字，存为列表
words= open('D:\\Vault-4\\Projects\\makemore\\names.txt', 'r').read().splitlines()
print(len(words))              # 数据集中总共有多少个名字（32033个）
print(max(len(w) for w in words)) # 最长的名字有多少个字符（15个）
print(words[:8])               # 预览前8个名字，直观感受数据长什么样

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [4]:
# --- 构建字符 ↔ 索引的双向映射表 ---
chars= sorted(list(set(''.join(words)))) # 将所有名字拼接→提取不重复字符→排序，得到 a-z 共26个字母
stoi= {s:i+1 for i,s in enumerate(chars)} # 字符→索引：a=1, b=2, ..., z=26（预留0给特殊字符）
stoi['.']=0                               # '.' 作为序列的起始/结束标记，索引设为 0
itos= {i:s for s,i in stoi.items()}       # 索引→字符：反向映射，用于把模型输出转回可读字符
vocab_size= len(stoi)                     # 词表大小 = 27（26个字母 + 1个特殊标记'.'）
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [ ]:
# --- 构建训练/验证/测试数据集 ---
block_size= 8  # 上下文窗口大小：用前 8 个字符预测下一个字符（比 Lecture 4 的 3 扩大了）

def build_dataset(words):
    """
    将名字列表转换为训练样本 (X, Y)。
    采用滑动窗口方式：每次取 block_size 个字符作为输入，下一个字符作为目标。
    例如 "tauren" → [........]→t, [.......t]→a, [......ta]→u, ...
    """
    X, Y= [], []
    for w in words:
        context= [0]*block_size       # 初始上下文全为 0（即 '........'），表示序列开头
        for ch in w+'.':              # 遍历名字的每个字符，末尾加 '.' 标记结束
            ix= stoi[ch]              # 当前字符转为索引
            X.append(context)         # 当前上下文作为输入
            Y.append(ix)             # 当前字符作为预测目标
            context= context[1:]+[ix] # 滑动窗口：丢弃最左边字符，右边加入新字符
    X= torch.tensor(X)               # 转为 PyTorch 张量，形状 (样本数, block_size)
    Y= torch.tensor(Y)               # 转为 PyTorch 张量，形状 (样本数,)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)                       # 固定随机种子，确保每次运行划分结果一致
random.shuffle(words)                 # 打乱名字顺序，避免数据有序带来的偏差
n1= int(0.8*len(words))              # 80% 数据用于训练
n2= int(0.9*len(words))              # 10% 用于验证（dev），剩余 10% 用于测试

Xtr, Ytr= build_dataset(words[:n1])   # 训练集：约 18 万个训练样本
Xdev, Ydev= build_dataset(words[n1:n2]) # 验证集：用于调超参数
Xte, Yte= build_dataset(words[n2:])   # 测试集：最终评估模型性能

In [ ]:
# --- 预览训练样本：8 字符上下文 → 预测下一个字符 ---
# 注意开头的 '.' 填充如何逐步被真实字符替换
for x, y in zip(Xtr[:20], Ytr[:20]):
    print(''.join(itos[i.item()] for i in x), '→', itos[y.item()])

In [92]:
# ============================================================
# 用类封装的深层网络：5 层隐藏层 + BatchNorm1d
# 架构：Embedding → [Linear → BN → Tanh] × 5 → Linear → Softmax
# 这是"正式"写法，替代前面手写的单层 BN
# ============================================================

class Linear:
    """线性层：y = x @ W + b"""
    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn((fan_in, fan_out)) / (fan_in**0.5)  # Kaiming 初始化, 这本来是可以保持。但是后面用了tanh，所以乘以 5/3 补偿
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x):
        self.out = x @ self.weight       # 矩阵乘法
        if self.bias is not None:
            self.out += self.bias
        return self.out
    
    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])
    

class BatchNorm1d:
    """批归一化层：训练时用 mini-batch 统计量，推理时用 running 统计量"""
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps                   # 防止除零的小常数
        self.momentum = momentum         # EMA 动量（0.1 → 近 ~10 个 batch 的平均）
        self.training = True             # 训练/推理模式开关
        # 不存 self.dim：dim 只用于下面创建张量的形状，
        # 之后维度信息已编码在张量 shape 中，__call__ 靠广播自动匹配，无需再引用

        # 可学习参数（通过反向传播更新）
        self.gamma = torch.ones((1, dim))   # 缩放因子 γ，初始 = 1
        self.beta = torch.zeros((1, dim))   # 偏移量 β，初始 = 0

        # 缓冲区（通过 EMA 更新，不参与梯度计算）
        self.running_mean = torch.zeros((1, dim))
        self.running_var = torch.ones((1, dim))

    def __call__(self, x):
        # ---- 1. 根据模式选择统计量来源 ----
        if self.training:
            if x.ndim==2:
                dim=0
            elif x.ndim==3:
                dim=0,1
            # 训练：用当前 mini-batch 实时计算（有噪声，但噪声起正则化作用）
            xmean = x.mean(dim, keepdim=True)                    # batch 均值 (1, dim)
            xvar = x.var(dim, keepdim=True, unbiased=False)       # batch 方差，用 1/N 非 1/(N-1)，与 PyTorch 官方 BN 一致
        else:
            # 推理：用 EMA 积累的全局统计量，保证同一输入每次输出一致
            xmean = self.running_mean
            xvar = self.running_var

        # ---- 2. BN 公式：标准化 → 缩放偏移 ----
        # xhat = (x - μ) / √(σ² + ε)  → N(0,1)
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)       # +eps 防止方差为 0 时除零
        # out = γ * xhat + β  → N(β, γ²)，让网络自己学最优分布
        self.out = self.gamma * xhat + self.beta

        # ---- 3. 训练时用 EMA 更新 running 统计量（簿记，不参与梯度） ----
        # 每步以 momentum 权重混入当前 batch 统计量，训练越久越接近全局真实分布
        if self.training:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
        return self.out
    
    def parameters(self):
        # 只有 γ 和 β 是可学习参数；running_mean/var 是 EMA 缓冲区，不参与梯度更新
        return [self.gamma, self.beta]
    

class Tanh:
    """Tanh 激活函数层"""
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out
    
    def parameters(self):
        return []
    
class Embedding:
    """嵌入层：将离散索引映射到连续向量空间"""
    def __init__(self, vocab_size, embedding_dim):
        self.weight = torch.randn((vocab_size, embedding_dim)) 

    def __call__(self, IX):
        # x 是索引张量，形状 (batch_size, block_size)
        # 输出是对应嵌入向量，形状 (batch_size, block_size, embedding_dim)
        self.out = self.weight[IX]  # 利用 PyTorch 的高级索引功能
        return self.out
    
    def parameters(self):
        return [self.weight]
    
class FlattenConsecutive:
    def __init__(self, n):
        self.n = n  # 每 n 个连续维度展平为一个维度

    def __call__(self, x):
        B, T, C = x.shape  # 假设输入形状为 (batch_size, block_size, embedding_dim)
        x=x.view(B, T//self.n, C*self.n) # 每 n 个连续维度展平为一个维度，新的形状为 (batch_size, block_size//n, embedding_dim*n)
        if x.shape[1] == 1: # 如果 block_size//n=1，说明已经展平到最后了，可以去掉那个维度
            x= x.squeeze(1) # 去掉 block_size//n=1 的维度，新的形状为 (batch_size, embedding_dim*n)
        self.out = x
        return self.out
    
    def parameters(self):
        return []
    
class Sequential:
    """顺序容器：将多个层串联起来，依次调用"""
    def __init__(self, layers):
        self.layers = layers
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)  # 依次通过每一层
        self.out = x
        return x
    
    def parameters(self):
        # 汇总所有层的可学习参数
        return [p for layer in self.layers for p in layer.parameters()]

## 2. 网络架构：WaveNet 风格的层级融合

### 核心思想：从"全部拍平"到"逐层融合"

Lecture 4 的 MLP 做法很粗暴：把 3 个嵌入向量直接拼接成一个长向量 `(30,)`，一次性喂进全连接层。这就像阅读时**把所有字母同时看完再理解**——效率低，也不符合直觉。

WaveNet 的做法更聪明：**先让相邻的字符两两融合，再让融合后的结果继续两两融合**，逐层向上汇聚，形成一棵二叉树：

```
层级3 (最终):          [h₁₂₃₄₅₆₇₈]          → (B, 200) 全局特征
                      /            \
层级2:         [h₁₂₃₄]            [h₅₆₇₈]    → (B, 2, 200)
              /      \            /      \
层级1:     [h₁₂]    [h₃₄]    [h₅₆]    [h₇₈]  → (B, 4, 200)
           / \      / \      / \      / \
层级0:    e₁  e₂  e₃  e₄  e₅  e₆  e₇  e₈    → (B, 8, 24) 嵌入
```

**生活类比：** 就像体育比赛的淘汰赛——8 个选手先两两对决（第 1 轮），胜出的 4 个再两两对决（第 2 轮），直到决出冠军。每一轮都在"融合信息"，而不是让 8 个人同时打成一团。

### `FlattenConsecutive(2)` 的作用

这是实现"两两融合"的关键操作。它把相邻的 2 个 token 的特征拼接起来：

$$\text{(B, 8, 24)} \xrightarrow{\text{FC(2)}} \text{(B, 4, 48)} \xrightarrow{\text{FC(2)}} \text{(B, 2, 400)} \xrightarrow{\text{FC(2)}} \text{(B, 400)}$$

每次序列长度减半，特征维度翻倍——相邻 token 的信息被压缩到同一个向量里，然后由 `Linear` 层学习如何提取有用的组合特征。

> **💡 深入理解：** `FlattenConsecutive` 和 1D 卷积（kernel_size=2, stride=2）做的事情本质一样——都是把相邻元素的特征拼起来再做线性变换。区别只是卷积用的是共享权重的滑动窗口，而这里是不重叠的分组。这也是为什么 Karpathy 把这个架构叫做 "WaveNet 风格"——WaveNet 论文中正是用空洞卷积（dilated convolution）实现了类似的层级融合结构。

In [93]:
torch.manual_seed(42)  # 固定随机种子，确保每次运行结果一致

In [102]:
n_embd = 24   # 嵌入维度：每个字符用 24 维向量表示
n_hidden = 200 # 隐藏层大小：每层隐藏层有 200 个神经元
# block_size=8, FC(2) 减半 3 次: 8→4→2→1(squeeze)→2D
# 第 4 组不需要 FlattenConsecutive（已经是 2D），直接用 Linear(200→200) 加深网络
model=Sequential([
    Embedding(vocab_size, n_embd), 
    FlattenConsecutive(2),Linear(n_embd*2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    FlattenConsecutive(2),Linear(n_hidden*2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    FlattenConsecutive(2),Linear(n_hidden*2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, vocab_size)
])
with torch.no_grad():
    model.layers[-1].weight *= 0.1 # 最后一层权重乘以 0.1，避免初始输出过大导致 loss NaN

parameters= model.parameters() # 汇总所有可学习参数
print(f'参数总数: {sum(p.numel() for p in parameters)}')
for p in parameters:
    p.requires_grad = True  # 标记所有参数需要计算梯度

参数总数: 217275


In [ ]:
# ============================================================
# 训练循环：200k 步 SGD，阶梯式学习率衰减
# ============================================================
max_steps=200000 
batch_size=32 
lossi=[] 
for i in range(max_steps): 
    # --- 1. 采样 mini-batch ---
    ix=torch.randint(0,Xtr.shape[0],(batch_size,))
    Xb,Yb =Xtr[ix],Ytr[ix]                     # (32, 8), (32,)

    # --- 2. 前向传播 ---
    logits=model(Xb)                             # (32, 27) 经过整个网络得到 logits
    loss=F.cross_entropy(logits,Yb)              # 交叉熵损失

    # --- 3. 反向传播 ---
    for p in parameters:
        p.grad = None                            # 清空旧梯度
    loss.backward()                              # 自动求导，计算所有参数的梯度

    # --- 4. 参数更新：SGD + 阶梯式学习率衰减 ---
    lr= 0.1 if i<150000 else 0.01                # 前 150k 步用大学习率，后 50k 步用小学习率微调
    for p in parameters:
        p.data += -lr * p.grad

    # --- 5. 记录统计信息 ---
    if i %10000 ==0:
        print(f'i:{i:7d}/{max_steps:7d}：{loss.item():.4f}')
    lossi.append(loss.log10().item())             # 记录 log10(loss) 方便可视化

## 3. 训练

标准的 SGD 训练循环，200k 步，阶梯式学习率衰减（前 150k 步 lr=0.1，后 50k 步 lr=0.01）。

与 Lecture 4 的训练完全相同，区别只在模型结构。这也是模块化设计的好处——`Sequential` 容器让我们可以随意替换网络架构，而训练代码一行不用改。

In [ ]:
# 将 loss 曲线按每 1000 步取均值后绘图，平滑掉 mini-batch 噪声
plt.plot(torch.tensor(lossi).view(-1,1000).mean(1)) 

In [ ]:
# 切换到推理模式：BN 层改用 EMA 统计量（running_mean/var），不再依赖当前 batch
for layer in model.layers:
    layer.training = False

## 4. 评估与采样

训练完成后，需要做两件事：
1. **切换到推理模式** —— BN 层不再用当前 batch 的统计量，而是用训练中 EMA 积累的全局统计量
2. **对比 train/dev loss** —— 判断模型是否过拟合

> **为什么推理时 BN 行为不同？** 训练时用 mini-batch 均值/方差有"噪声"，这个噪声其实起了正则化作用（类似 Dropout）。但推理时我们希望同一输入永远得到同一输出——所以用 EMA 积累的稳定统计量。

In [ ]:
# --- 评估训练集和验证集的损失 ---
@torch.no_grad()                                 # 关闭梯度追踪，节省内存
def split_loss(split):
    X,Y = {'train':(Xtr,Ytr),'dev':(Xdev,Ydev),'test':(Xte,Yte)}[split]
    logits=model(X)                              # 对整个数据集做前向传播
    loss=F.cross_entropy(logits,Y)               # 计算平均交叉熵损失
    print(f'{split} loss: {loss.item():.4f}')

split_loss('train')
split_loss('dev')

## Performance Logs

### 1. 增大 block_size=8
- train loss: 1.9208
- dev loss: 2.0281

### 2. FlattenConsecutive 展平
- train loss: 1.9395
- dev loss: 2.0350

### 3. 修复 BatchNorm bug
- train loss: 1.9123
- dev loss: 2.0293

### 4. 增大 n_embd=24
- train loss: 1.7651
- dev loss: 2.0007

### 5. 增加第 4 层
- train loss: 1.6399
- dev loss: 2.0335

In [ ]:
# --- 从模型中采样生成名字 ---
for _ in range(20):
    out = []
    context = [0] * block_size                   # 初始上下文全为 '.'
    while True:
        logits=model(torch.tensor([context]))     # (1, 27) 单样本前向传播
        probs = F.softmax(logits, dim=1)          # 转为概率分布
        ix = torch.multinomial(probs, num_samples=1).item()  # 按概率采样下一个字符
        out.append(itos[ix])                      # 索引→字符
        context = context[1:] + [ix]              # 滑动窗口更新上下文
        if ix == 0:                               # 遇到 '.' 表示名字结束
            break
    print(''.join(out))